# KArSL QC and Split Verification

Read-only verification of packaged QC classifications and signer-independent folds. This notebook checks saved artifacts; it does not recreate QC from videos.


In [ ]:
from pathlib import Path
import os
import pandas as pd
root = Path(os.environ.get("SIGNOVA_KARSL_ROOT", ".")).expanduser().resolve()
if not (root / "data/metadata/karsl_metadata.csv").is_file():
    candidate = Path.cwd() / "karsl"
    if (candidate / "data/metadata/karsl_metadata.csv").is_file():
        root = candidate.resolve()
    else:
        raise FileNotFoundError("Set SIGNOVA_KARSL_ROOT to the unpacked karsl folder")
print("KArSL root:", root)


In [ ]:
m = pd.read_csv(root / "data/metadata/karsl_metadata.csv", low_memory=False)
s = pd.read_csv(root / "data/splits/karsl_split_manifest.csv", low_memory=False)
pass_ids = set(m.loc[m["qc_status"].eq("PASS"), "sample_id"].astype(str))
active_ids = set(s["sample_id"].astype(str))
checks = {
    "Metadata row count": len(m) == 75514,
    "PASS count": int(m["qc_status"].eq("PASS").sum()) == 75296,
    "EXCLUDE count": int(m["qc_status"].eq("EXCLUDE").sum()) == 150,
    "REVIEW count": int(m["qc_status"].eq("REVIEW").sum()) == 68,
    "Three folds": s["fold"].nunique() == 3,
    "Only PASS samples in folds": active_ids.issubset(pass_ids),
    "Expected benchmark IDs": len(active_ids) == 75095,
    "No duplicate IDs within folds": not s.duplicated(["fold", "sample_id"]).any(),
    "Same benchmark IDs per fold": all(set(g["sample_id"]) == active_ids for _, g in s.groupby("fold")),
    "No signer overlap within folds": all(g.groupby("signer_id")["split"].nunique().max() == 1 for _, g in s.groupby("fold")),
    "500 benchmark classes": s["sign_id"].nunique() == 500
}
for name, ok in checks.items():
    print(name, "PASS" if ok else "FAIL")
assert all(checks.values())
print("Semantic labels independently confirmed: NO")
